# Agilent BioTek Cytation 7

```{device-card} biotek-cytation-7
```

| Property | Value |
| --- | --- |
| Connection | Direct USB through PLR's FTDI transport |
| Python extra | `ftdi` |
| Interface | `pylabrobot.agilent.biotek.cytation.Cytation7` |
| Hardware verification | Pending for this class and its inherited reader operations |
| Microscopy | Not initialized or supported by this class |
| Temperature setting | Disabled; current-temperature queries are available |

```{warning}
This implementation has not been verified on Cytation 7 hardware. It inherits the
shared BioTek reader's commands, defaults, and limits. These are not established C7
specifications. An owner has reported tray in/out and current-temperature queries
working on a physical C7 through a Cytation5 interface; that installation's exact
version/import path and firmware remain to be recorded.

Execute only the cells covered by an approved hardware session, not Run All.
Setup resets USB and is itself a hardware operation. Stop after any unexpected
response; do not automatically retry an acquisition or move a tray in unknown state.
```

The offline tests exercise inherited and synthetic responses through mocked
transport. They do not establish measurement accuracy or Gen5 configuration parity.

## Connection and physical setup

Install the FTDI extra in a separate development environment from the working lab
installation: `python -m pip install "pylabrobot[ftdi]"`. From a source checkout,
use `python -m pip install -e ".[ftdi]"`. Follow the
[PLR installation guide](../../getting-started/installation.md) for transport setup.
Preserve the lab's working USB driver configuration; the Python extra alone does
not establish that system USB libraries/drivers are ready.

Connect to the plate-reader USB interface, confirm the physical C7 and its actual
FTDI serial identifier, and ensure the carrier area is clear. The serial in a Gen5
report is not necessarily the FTDI identifier. Release Gen5's connection before
transferring USB to the PLR computer. Use an exact, reviewed plate definition and
lid/seal configuration before closing a loaded tray.

Review the [hardware validation checklist](validation.md) before the session.

## Create the reader

Enter the actual FTDI identifier approved for this session. Construction does not
open USB. C7 shares the reader base with C1/C5 and does not initialize a microscope.

In [ ]:
from pylabrobot.agilent.biotek.cytation import Cytation7

device_id = ""
if not device_id.strip():
  raise ValueError("Enter the actual FTDI identifier for the approved Cytation 7.")

reader = Cytation7(name="cytation7", device_id=device_id)

## Start recording

Start PLR's native capture after constructing the reader and before setup. Each
session uses a separate file. Capture records transport settings and full read/write
bytes, including firmware replies; it does not record Gen5 traffic or provide
per-operation timestamps. Keep a separate timestamped session record.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from pylabrobot.io.capture import start_capture, stop_capture

capture_dir = Path("cytation7-captures")
capture_dir.mkdir(exist_ok=True)
session_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
capture_path = capture_dir / f"{session_stamp}-plr.json"
if capture_path.exists():
  raise FileExistsError(capture_path)
start_capture(capture_path)

## Connect

Setup opens and resets USB, configures the shared transport, and queries firmware.
It tries 9600 baud and retries the firmware query at 38461 baud after a timeout.
These are inherited settings, not independently measured C7 requirements.

If setup fails, release the transport directly and save the capture. The inherited
`stop()` path assumes setup has initialized the shaking task state. Do not continue
into operation cells after setup failure.

In [ ]:
try:
  await reader.setup()
except Exception:
  try:
    await reader.io.stop()
  finally:
    stop_capture()
  raise

## Read instrument identity

Record the returned instrument serial separately from the FTDI identifier.

In [ ]:
instrument_serial = await reader.request_serial_number()
instrument_serial

## Read firmware

This method returns a parsed version token. Retain the complete reply in the
capture for basecode identification; `reader.version` contains the setup-time token.

In [ ]:
firmware_version = await reader.request_firmware_version()
firmware_version

## Read current temperature

The parsed value uses degrees Celsius. Compare it with Gen5 and retain the raw reply.

In [ ]:
temperature = await reader.request_current_temperature()
temperature

## Open the tray

Confirm the carrier area is clear. `slow=True` requests the existing reduced-speed
mode; its C7 behavior also needs verification. Start with an approved empty-tray test.

In [ ]:
await reader.open(slow=True)

## Define the plate

Replace `None` with the PLR definition for the exact plate and lid approved for this
session. A generic 96-well definition is not sufficient. The lab's reported Greiner
uClear Black plate still needs a catalogue number and geometry confirmation; no
substitute geometry is supplied here.

In [ ]:
from typing import Optional

from pylabrobot.resources import Plate

plate: Optional[Plate] = None
if plate is None:
  raise ValueError("Supply the exact reviewed plate and lid definition before loading.")

## Load and close the tray

Place the approved plate in the reviewed orientation, then execute the next cell.
`close(plate=...)` sends plate geometry before closing. Calling `close()` without a
plate is for an empty tray; it does not send geometry. `reader.plate_holder` is the
inherited resource holder, not a sensor confirming the physical loading state.

In [ ]:
await reader.close(plate=plate, slow=True)

## Select wells

Begin with a single approved well and known-position controls. Use
`plate.get_all_items()` only after full-plate acquisition and orientation are approved.

In [ ]:
wells = plate["A1"]

## Read absorbance at 600 nm

The API accepts one wavelength per call. Read speed, delay, and measurements per
point are not configurable here; matching Gen5's settings remains to be established.
Result data is indexed `[row][column]`, with `None` for omitted wells and `NaN` for
asterisk-valued records. The device meaning of those asterisks needs verification.

In [ ]:
absorbance_600 = await reader.read_absorbance(plate=plate, wells=wells, wavelength=600)
absorbance_600[0].data

## Read absorbance at 700 nm

This is a separate acquisition, not a verified equivalent of a single Gen5
multi-wavelength step. Keep the two channels separate; no subtraction is implied.

In [ ]:
absorbance_700 = await reader.read_absorbance(plate=plate, wells=wells, wavelength=700)
absorbance_700[0].data

## Read fluorescence

The example centres and height come from the lab's target workflow. Execute only
after reviewing the optical path and plate geometry for this specific test. The
numeric call does **not** select verified bottom optics, 20 nm bandwidths, Filter
Set 1, or gain 85. Those controls are not exposed, and C7 height calibration is
unverified. This is not a validated GFP assay.

In [ ]:
fluorescence = await reader.read_fluorescence(
  plate=plate,
  wells=wells,
  excitation_wavelength=485,
  emission_wavelength=530,
  focal_height=7,
)
fluorescence[0].data

## Read luminescence

Luminescence is included in the inherited API, although it is not part of the lab's
target assay. Use an installed compatible optical module and appropriate stable
controls. Review the example 7 mm height and 1 second integration before execution.

In [ ]:
luminescence = await reader.read_luminescence(
  plate=plate, wells=wells, focal_height=7, integration_time=1
)
luminescence[0].data

## Temperature-control availability

`set_temperature()` raises `NotImplementedError` before I/O because inherited heating
and cooling flags are disabled. This is a software limitation, not a statement about
the C7 hardware. Gradient and bounded preheating are follow-up work. The inherited
`stop_heating_or_cooling()` and `deactivate()` methods send a temperature-off command;
only use them when that command and final heat state are explicitly approved.

In [ ]:
reader.supports_heating, reader.supports_cooling

## Choose an existing shaking mode

Only `LINEAR` and `ORBITAL` are exposed. The inherited `frequency` argument is a
selector from 1 through 6; do not pass 365 or infer a C7 cpm/amplitude mapping from
the legacy documentation. Double orbital is not implemented. Fill in the setting
agreed for the plate, volume, and lid. Do not run acquisition concurrently with shaking.

In [ ]:
shake_type = Cytation7.ShakeType.ORBITAL
shake_setting: Optional[int] = None
if shake_setting is None or not 1 <= shake_setting <= 6:
  raise ValueError("Choose an approved legacy shaking selector from 1 through 6.")

## Perform a bounded shaking trial

Include the five-second startup timeout, five-second observation, and stop command
in the run approval. The helper runs a background task that downloads repeated
16-minute assays; this example requests a stop after a short observation. Host-side
cancellation alone does not prove physical stopping. A failed startup can leave a
background task or unknown device state; record the outcome and stop the session
if stopping cannot be confirmed.

In [ ]:
import asyncio

try:
  await asyncio.wait_for(reader.shake(shake_type, frequency=shake_setting), timeout=5)
  await asyncio.sleep(5)
finally:
  await reader.stop_shaking()

## Homing and geometry helpers

`home()` is inherited but is not part of the first identity/tray session. Its C7
motion and recovery semantics need separate qualification. `set_plate(plate)` sends
geometry without closing; `clear_plate()` only forgets the cached geometry, forcing
it to be resent at the next applicable operation. Neither confirms physical position.

## Release USB and save the capture

After a successfully initialized session, stop the reader before transferring USB
back to the Gen5 computer. `stop()` stops an active shaking task and releases USB;
it does not eject the plate or set a final temperature. Approve any such operations
separately. Save the capture even if shutdown fails, and confirm physical stopping
and clean Gen5 handback with the operator.

In [ ]:
try:
  await reader.stop()
finally:
  stop_capture()

## Hardware qualification

```{toctree}
:maxdepth: 1

validation
```